## Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or python modules that might be required to run this notebook.

# Fetching Live Quotes with KiteConnect

Once authenticated and with access to the instrument list, the next fundamental task is to get price information. The KiteConnect API provides several ways to fetch quotes, each serving a different purpose. Understanding these formats is key to building effective trading tools.

This notebook will cover the three primary methods for fetching quotes:

1.  **LTP (Last Traded Price):** Get the most recent price at which an instrument was traded.
2.  **OHLC (Open, High, Low, Close):** Get a summary of the day's trading activity for an instrument.
3.  **Full Quote:** Get a comprehensive view of an instrument, including market depth (best bids and offers), daily statistics, and more.

## Prerequisites

* **Pandas Library**: For displaying the quote data in a readable format.

## Step 1: Authentication

As always, our first step is to get an authenticated `kite` object using the following code.

In [ ]:
# Import the KiteConnect class from the kiteconnect package to interact with Zerodha's Kite API
from kiteconnect import KiteConnect

import pandas as pd

class KiteAuth:
    """A class to handle the KiteConnect authentication flow."""
    def __init__(self, api_key, api_secret):
        """
        Initialises the KiteAuth object with API credentials.

        Parameters:
        - api_key: Your Zerodha Kite API key
        - api_secret: Your Zerodha Kite API secret
        """
        self.api_key = api_key  # Store the API key
        self.api_secret = api_secret  # Store the API secret
        self.kite = None  # Will hold the authenticated KiteConnect object

    def authenticate(self):
        """
        Guides the user through the manual authentication process.

        Steps:
        - Initialises KiteConnect
        - Displays the login URL to the user
        - Prompts the user to input the request token received after login
        - Uses the token to generate a session and retrieve the access token
        - Sets the access token to the KiteConnect instance
        - Returns the authenticated KiteConnect object if successful
        """
        try:
            # Create a KiteConnect instance with the given API key
            kite = KiteConnect(api_key=self.api_key)

            # Instruct the user to log in via the URL to generate a request token
            print("Please click on the URL below to login:")
            print(kite.login_url())
            print("=" * 50)

            # Accept the request token manually from the user
            request_token = input(
                "Enter the request token from redirect URL: ").strip()

            # Generate a session using the request token and API secret
            data = kite.generate_session(request_token,
                                         api_secret=self.api_secret)

            # Set the access token in the KiteConnect instance for future API calls
            kite.set_access_token(data['access_token'])

            # Save the authenticated KiteConnect instance
            self.kite = kite

            # Print a welcome message with the user's name
            print("=" * 50)
            print("Authentication successful!")
            print(f"Welcome!")
            print("=" * 50)

            return kite  # Return the authenticated KiteConnect instance

        except Exception as e:
            # Handle errors that may occur during authentication
            print(f"Authentication failed: {e}")
            return None


# --- Start Authentication ---
# IMPORTANT: Replace with your credentials
API_KEY = "YOUR_API_KEY"
API_SECRET = "YOUR_API_SECRET"

kite = None
if API_KEY != "YOUR_API_KEY":
    # This is how you would use the imported class
    auth = KiteAuth(API_KEY, API_SECRET)
    kite = auth.authenticate()
else:
    print(
        "\nPlease update your API_KEY and API_SECRET to run the authentication."
    )

Please click on the URL below to login:
https://kite.zerodha.com/connect/login?api_key=tj19jzf6vgdl162v&v=3
Enter the request token from redirect URL: mNpqG0vnbWZrx5mm8rkkzruL9PnuwkXV
Authentication successful!
Welcome!


## Step 2: Define Instruments

We need a list of instruments to fetch quotes for. The instrument symbols must be in the format `EXCHANGE:TRADINGSYMBOL`.

In [ ]:
instrument_list = [
    'NSE:INFY',
    'NSE:RELIANCE',
    'BSE:TCS',
    'NSE:NIFTY 50' # An index
]

## Understanding Quote Formats

Let's now fetch the quotes for our list of instruments in all three formats.

### 1. Last Traded Price (LTP)

This is the quickest way to get the latest price. The `kite.ltp()` method returns a dictionary where keys are the instrument symbols and values are dictionaries containing the `instrument_token` and `last_price`.

In [ ]:
if kite:
    try:
        ltp_data = kite.ltp(instrument_list)
        print("--- LTP (Last Traded Price) ---")
        print(pd.DataFrame(ltp_data).T) # Transpose for better readability
    except Exception as e:
        print(f"Error fetching LTP: {e}")

--- LTP (Last Traded Price) ---
              instrument_token  last_price
BSE:TCS            136330244.0     3424.80
NSE:INFY              408065.0     1616.90
NSE:NIFTY 50          256265.0    24802.85
NSE:RELIANCE          738561.0     1435.40


### 2. Open, High, Low, Close (OHLC)

OHLC provides a summary of the day's price movement. It gives you the opening price, the highest price reached, the lowest price reached, and the previous day's closing price. The `kite.ohlc()` method works similarly to the LTP call.

In [ ]:
if kite:
    try:
        ohlc_data = kite.ohlc(instrument_list)
        print("\n--- OHLC (Open, High, Low, Close) ---")
        print(pd.DataFrame(ohlc_data).T)
    except Exception as e:
        print(f"Error fetching OHLC: {e}")


--- OHLC (Open, High, Low, Close) ---
             instrument_token last_price  \
BSE:TCS             136330244     3424.8   
NSE:INFY               408065     1617.1   
NSE:NIFTY 50           256265   24804.05   
NSE:RELIANCE           738561     1435.2   

                                                           ohlc  
BSE:TCS       {'open': 3448.7, 'high': 3452, 'low': 3413.35,...  
NSE:INFY      {'open': 1623.9, 'high': 1630, 'low': 1611.5, ...  
NSE:NIFTY 50  {'open': 24803.25, 'high': 24863.1, 'low': 247...  
NSE:RELIANCE  {'open': 1430.1, 'high': 1437, 'low': 1425.7, ...  


### 3. Full Quote

The full quote provides the most comprehensive data packet. It includes the OHLC, LTP, total volume, average traded price, and crucially, the **market depth**. Market depth shows the top 5 pending buy orders (bids) and sell orders (asks) in the market, giving you insight into the current supply and demand.

The `kite.quote()` method returns a dictionary that contains all of this information.

In [ ]:
if kite:
    try:
        quote_data = kite.quote(instrument_list)
        print("\n--- Full Market Quote ---")

        # The result is rich with data, let's process it for one instrument
        for instrument, data in quote_data.items():
            print(f"\n--- Quote for {instrument} ---")
            print(f"Last Price: {data['last_price']}")
            print(f"Volume: {data['volume']}")
            print(f"Average Price: {data['average_price']}")
            print(f"OHLC: O={data['ohlc']['open']} H={data['ohlc']['high']} L={data['ohlc']['low']} C={data['ohlc']['close']}")

            # Displaying market depth (top 5 bids and asks)
            print("\nMarket Depth:")
            depth = {'bids': data['depth']['buy'][:5], 'asks': data['depth']['sell'][:5]}
            bids_df = pd.DataFrame(depth['bids'])
            asks_df = pd.DataFrame(depth['asks'])
            depth_df = pd.concat([bids_df.rename(columns={'price': 'bid_price', 'quantity': 'bid_qty'}),
                                  asks_df.rename(columns={'price': 'ask_price', 'quantity': 'ask_qty'})], axis=1)
            display(depth_df.fillna('-'))
            print("-"*40)

    except Exception as e:
        print(f"Error fetching Full Quote: {e}")


--- Full Market Quote ---

--- Quote for BSE:TCS ---
Last Price: 3424.8
Volume: 34746
Average Price: 3427.99
OHLC: O=3448.7 H=3452 L=3413.35 C=3452.2

Market Depth:


,bid_price,bid_qty,orders,ask_price,ask_qty,orders
0,3423.00,37,2,3425.20,45,5
1,3422.70,175,1,3425.25,22,4
2,3422.60,6,1,3425.30,23,4
3,3422.55,5,1,3425.35,58,7
4,3422.40,3,1,3425.40,30,4


----------------------------------------

--- Quote for NSE:INFY ---
Last Price: 1617.1
Volume: 2225111
Average Price: 1617.97
OHLC: O=1623.9 H=1630 L=1611.5 C=1632.9

Market Depth:


,bid_price,bid_qty,orders,ask_price,ask_qty,orders
0,1617.1,56,1,1617.3,767,4
1,1616.9,364,4,1617.4,179,4
2,1616.8,256,9,1617.5,619,9
3,1616.7,296,4,1617.6,336,11
4,1616.6,249,6,1617.7,1130,9


----------------------------------------

--- Quote for NSE:NIFTY 50 ---
Last Price: 24803.95
Error fetching Full Quote: 'volume'


## When to Use Each Format

* **LTP**: Best for dashboards, watchlists, or any application where you just need a quick, lightweight price update.
* **OHLC**: Useful for daily analysis, candlestick chart data, and understanding the day's trading range.
* **Full Quote**: Essential for order placement logic. Analyzing the market depth (the spread between bid and ask prices and the quantities available) helps in making decisions about placing limit vs. market orders and assessing liquidity.

Note that there will be times when you get an error. You can check the documentation at the following link: <a href="https://kite.trade/docs/connect/v3/exceptions/" target="_blank" rel="noopener">Errors and Exceptions</a>
 to understand what type of error it is.

## Conclusion

You can now programmatically fetch live market data in three different formats using the KiteConnect API. These functions are based on a polling mechanism, meaning you have to call them every time you want a fresh update.

You can get tick-by-tick data without constantly making API calls, the next step is to explore the **KiteConnect WebSocket API**, which streams data directly to your application. We will cover this in a future notebook. <br><br>